### 总结中间件 SummarizationMiddleware

当对话历史变长、接近模型上下文窗口时，`SummarizationMiddleware` 会把较早的消息自动总结成一条摘要消息，从而压缩上下文，同时保留最近的一些消息。

- `trigger`：触发压缩的条件，支持多种写法，满足任意一条即触发（列表是 **OR**；同一个 `dict` 里的多个条件是 **AND**）
  - `("tokens", 100)`：估算 token 数 ≥ 100
  - `("messages", 6)`：消息条数 ≥ 6
  - `("fraction", 0.001)`：达到模型最大输入窗口的 0.1%
- `keep`：压缩后保留多少最近的消息（不支持多个值）

> 注意：`fraction` 需要模型提供上下文窗口大小（`profile["max_input_tokens"]`），DeepSeek 默认没有该信息，所以要显式传入 `profile`。

#### `profile` 里的 `max_input_tokens` 是谁的？

`profile` 是挂在**模型对象**上的元数据，用来描述该模型的真实上下文窗口。

- 本示例中同一个 `model` 既负责回答、又被传给 `SummarizationMiddleware` 做总结，所以「回答模型」和「总结模型」是同一个，窗口也是同一个值。
- `fraction` 的阈值是按**传给中间件的那个模型**的 `profile` 计算的；如果用另一个模型专门做总结，要注意它的窗口可能不同。
- 这个值**不会改变模型真实窗口**，只是让 LangChain 组件知道边界，填错只会误导组件判断。
- **只有用 `fraction` 才需要 `profile`**，`("tokens", …)` / `("messages", …)` 不需要。

如何确定窗口大小：优先查服务商文档；LangChain 1.1+ 若能自动读到 `model.profile` 就无需手填，读不到（如本示例的 DeepSeek）时才需要手动传入。

In [8]:
## trigger， keep 参数
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage

load_dotenv(override=True)

# 关闭思考模式；并显式给出模型上下文窗口，fraction 触发条件才可用
model = init_chat_model(
    api_base=os.getenv("DEEPSEEK_API_BASE"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    model="deepseek-flash",
    model_provider="deepseek",
    # model_kwargs={"reasoning_effort": "none"},
    profile={"max_input_tokens": 65536},
)

# trigger 列表里任意一条满足即压缩：tokens>=100 或 messages>=6 或 达到窗口的 0.1%
# keep=("messages", 2) 表示压缩后只保留最近 2 条消息
agent = create_agent(
    model="deepseek:deepseek-flash",
    middleware=[
        SummarizationMiddleware(
            model,
            trigger=[
                ("tokens", 100),
                ("messages", 6),
                ("fraction", 0.001),
            ],
            keep=("messages", 2),
            summary_prompt="使用中文总结"
        )
    ],
)


In [9]:
# 查看模型 profile：只有能提供 max_input_tokens，fraction 触发才可用
print("带 profile 的模型：", model.profile)

# 不传 profile 时，构造带 fraction 的中间件会直接报错
model_no_profile = init_chat_model(
    api_base=os.getenv("DEEPSEEK_API_BASE"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    model="deepseek-flash",
    model_provider="deepseek",
)
try:
    SummarizationMiddleware(model_no_profile, trigger=[("fraction", 0.001)])
except ValueError as e:
    print("缺少 profile 报错：", str(e).splitlines()[0])


带 profile 的模型： {'max_input_tokens': 65536}
缺少 profile 报错： Model profile information is required to use fractional token limits, and is unavailable for the specified model. Please use absolute token counts instead, or pass `


#### 构造较长的对话历史并运行

In [10]:
# 构造一段较长的对话历史（7 轮问答，共 13 条消息），足以触发压缩
messages = [
    HumanMessage(content="你好，我想学习 LangChain。"),
    AIMessage(content="好的，LangChain 是一个用于构建大模型应用的框架，支持模型、提示词、工具、链等。"),
    HumanMessage(content="那第一个要学什么？"),
    AIMessage(content="建议先学模型初始化与调用，也就是 ChatModel 的基本用法。"),
    HumanMessage(content="模型调用有哪些方式？"),
    AIMessage(content="常见有 invoke、batch、stream 三种，分别对应同步、批量和流式。"),
    HumanMessage(content="流式有什么用？"),
    AIMessage(content="流式可以边生成边显示，提升交互体验，适合聊天场景。"),
    HumanMessage(content="那工具调用呢？"),
    AIMessage(content="工具调用可以让模型决定调用外部函数，比如查询天气、搜索网页等。"),
    HumanMessage(content="那中间件又是做什么的？"),
    AIMessage(content="中间件可以在模型调用前后插入逻辑，比如压缩上下文、注入内容、拦截工具等。"),
    HumanMessage(content="最后帮我总结一下学习路线。"),
]

# 运行 Agent：中间件会在调用模型前检查历史，满足 trigger 就先压缩
result = agent.invoke({"messages": messages})

# 压缩后会插入一条带 lc_source=summarization 标记的摘要消息，用它来判断是否发生了压缩
compressed = any(
    getattr(m, "additional_kwargs", {}).get("lc_source") == "summarization"
    for m in result["messages"]
)
if compressed:
    print("上下文已压缩")

print("压缩后消息数：", len(result["messages"]))
print("压缩后消息类型：", [type(m).__name__ for m in result["messages"]])
print(result["messages"])


上下文已压缩
压缩后消息数： 4
压缩后消息类型： ['HumanMessage', 'AIMessage', 'HumanMessage', 'AIMessage']
[HumanMessage(content='Here is a summary of the conversation to date:\n\n请提供需要总结的具体内容（文字、链接或文件），我才能用中文为你总结。', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='5d4b9f85-fcdf-4cfc-a00e-7f87d82c1c14'), AIMessage(content='中间件可以在模型调用前后插入逻辑，比如压缩上下文、注入内容、拦截工具等。', additional_kwargs={}, response_metadata={}, id='8a93082d-4338-41bf-852f-bf46994f7b3b', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='最后帮我总结一下学习路线。', additional_kwargs={}, response_metadata={}, id='1921518e-7744-43a2-818e-f1beb8b92931'), AIMessage(content='基于前面提到的“中间件可以在模型调用前后插入逻辑”，学习路线可以总结为：\n\n**一句话：先懂模型调用，再学中间件机制，最后用中间件解决上下文、注入、工具拦截等工程问题。**\n\n### 1. 基础阶段\n- 学会调用大模型 API\n- 理解 Prompt、Token、上下文窗口\n- 掌握 Function Calling / Tool Use\n- 理解 Agent 基本循环：思考 → 调工具 → 观察 → 再回答\n\n### 2. 中间件核心\n- 理解请求前、请求后、工具调用前后的钩子\n- 学习洋葱模型、责任链、拦截器、装饰器思想\n- 能自己写一个最小中间件管道：`before -> next -> after`\n\n### 3. 关键应用场景\n- 上下文管理：压缩、摘要、滑动窗口、RA

#### 查看压缩后的历史

In [12]:
# 压缩后的第一条就是生成的摘要，后面是保留的最近消息
print("========== 摘要（压缩后的上下文）==========")
print(result["messages"][0].pretty_print())

print("\n========== 保留的最近消息 ==========")
for m in result["messages"][1:]:
    print(f"[{type(m).__name__}]", str(m.content))


========== 摘要（压缩后的上下文）==========
================================ Human Message =================================

Here is a summary of the conversation to date:

请提供需要总结的具体内容（文字、链接或文件），我才能用中文为你总结。
None

========== 保留的最近消息 ==========
[AIMessage] 中间件可以在模型调用前后插入逻辑，比如压缩上下文、注入内容、拦截工具等。
[HumanMessage] 最后帮我总结一下学习路线。
[AIMessage] 基于前面提到的“中间件可以在模型调用前后插入逻辑”，学习路线可以总结为：

**一句话：先懂模型调用，再学中间件机制，最后用中间件解决上下文、注入、工具拦截等工程问题。**

### 1. 基础阶段
- 学会调用大模型 API
- 理解 Prompt、Token、上下文窗口
- 掌握 Function Calling / Tool Use
- 理解 Agent 基本循环：思考 → 调工具 → 观察 → 再回答

### 2. 中间件核心
- 理解请求前、请求后、工具调用前后的钩子
- 学习洋葱模型、责任链、拦截器、装饰器思想
- 能自己写一个最小中间件管道：`before -> next -> after`

### 3. 关键应用场景
- 上下文管理：压缩、摘要、滑动窗口、RAG 注入
- 内容注入：系统提示、记忆、用户画像、知识库
- 工具拦截：权限校验、参数校验、审计、重试、降级
- 输出处理：格式校验、敏感词过滤、结构化解析、缓存

### 4. 框架实战
- 选一个主流 Agent / LLM 框架深入
- 看它的 middleware、callbacks、filters、hooks 机制
- 把前面手写的能力迁移到框架里

### 5. 工程化进阶
- 可观测性：日志、Tracing、指标
- 评估：效果评估、回归测试
- 安全：权限、隔离、审计、限流
- 性能：缓存、并发、降级、重试

### 6. 项目驱动
做一个带中间件的迷你 Agent：
1. 基础对话
2. 加日志中间件
3. 加上下文压缩
4. 加 RAG

### 中间件内部是怎么工作的

每次调用模型前，`SummarizationMiddleware` 的 `before_model` 钩子会执行：

1. 给缺少 id 的消息补上唯一 id。
2. 用 `token_counter` 估算当前历史的 token 数，再由 `_should_summarize` 判断是否触发：
   - `trigger` 列表之间是 **OR**，同一个 `dict` 内多个条件是 **AND**；
   - `messages` 比对消息条数；`tokens`/`fraction` 既看估算值，也看上一条 AI 消息的**真实用量** `usage_metadata.total_tokens`；
   - `fraction` 的阈值 = `max_input_tokens × fraction`（本例 65536 × 0.001 ≈ 65）。
3. 触发后由 `_determine_cutoff_index` 依据 `keep` 计算切点，并保证不拆散 `AI(tool_calls)` 与 `ToolMessage` 的配对。
4. `_create_summary` 调用 `self._summary_model.invoke(...)` 生成摘要。
5. 返回新状态：`[RemoveMessage(REMOVE_ALL_MESSAGES), 摘要消息, *保留的消息]`。摘要是一条 `HumanMessage`，其 `additional_kwargs` 带 `lc_source="summarization"`——这就是我们用来判断「上下文已压缩」的标记。

一句话总结：**窗口大小由你通过 `profile` 告诉 LangChain；触发判断由「估算 token + 真实 usage」共同决定；`fraction` 只是把你的窗口按比例换算成阈值。**